In [1]:
import tensorflow as tf

C:\Users\PRAVALLIKA\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
print(tf.__version__)

2.21.0


In [3]:
import os
import cv2
import numpy as np

import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB0

from tensorflow.keras.layers import (
    Input,
    TimeDistributed,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.models import Sequential

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

# directory of dataset

In [4]:
dataset_dir = r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos"

# classes to predict

In [5]:
classes = [
    "Explosion",
    "fighting",
    #"Robbery",
    "Shooting",
    "z_Normal_Videos_event"
]

In [6]:
import os

dataset_dir = r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos"

for class_name in sorted(os.listdir(dataset_dir)):
    class_path = os.path.join(dataset_dir, class_name)

    # Check whether it is a folder
    if os.path.isdir(class_path):

        # Count video files
        videos = [
            file for file in os.listdir(class_path)
            if file.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
        ]

        print(f"{class_name} -> {len(videos)} files")

Abuse -> 50 files
Arrest -> 50 files
Explosion -> 50 files
Fighting -> 50 files
RoadAccidents -> 150 files
Robbery -> 150 files
Shooting -> 50 files
Shoplifting -> 50 files
Stealing -> 100 files
z_Normal_Videos_event -> 50 files


# Resize image

In [7]:
image_height = 224
image_width = 224
sequence_length = 20

# Loading pretrained model

In [8]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)

# freezing 

In [9]:
base_model.trainable = False

# creating dataset

In [10]:
def create_dataset():
    features=[]
    labels=[]
    video_files_paths=[]
    for class_index,class_name in enumerate(classes):
        print(f"extracting data of class {class_name}")
        files_list=os.listdir(os.path.join(dataset_dir,class_name))
        for file_name in files_list:
            video_file_path=os.path.join(dataset_dir,class_name,file_name)
            frames=frames_extraction(video_file_path)
            if len(frames)==sequence_length:
                features.append(frames)
                labels.append(class_index)
                video_files_paths.append(video_file_path)
    features=np.asarray(features)
    labels=np.array(labels)
    return features,labels,video_files_paths

# Extrcating frames from videos

In [11]:
def frames_extraction(video_path):

    frames_list = []

    video_reader = cv2.VideoCapture(video_path)

    video_frames_count = int(
        video_reader.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    skip_frames_window = max(
        int(video_frames_count / sequence_length),
        1
    )

    for frame_counter in range(sequence_length):

        video_reader.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_counter * skip_frames_window
        )

        success, frame = video_reader.read()

        if not success:
            break

        # BGR -> RGB
        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # Resize
        frame = cv2.resize(
            frame,
            (image_width, image_height)
        )

        frame = frame.astype(np.float32)

        frames_list.append(frame)

    video_reader.release()

    if len(frames_list) == sequence_length:
        return np.array(frames_list)

    return None

In [12]:
features, labels, video_files_paths = create_dataset()

extracting data of class Explosion
extracting data of class fighting
extracting data of class Shooting
extracting data of class z_Normal_Videos_event


# model building

In [13]:
model = Sequential([
    
    Input(shape=(20, 224, 224, 3)),

    TimeDistributed(base_model),

    LSTM(128),

    Dropout(0.4),

    Dense(64, activation="relu"),

    Dropout(0.3),

    Dense(len(classes), activation="softmax")
])

In [14]:
model.compile(
    optimizer=Adam(
        learning_rate=0.0001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [15]:
def data_generator(
    video_paths,
    labels,
    batch_size=4
):

    while True:

        indices = np.arange(
            len(video_paths)
        )

        np.random.shuffle(indices)

        for start in range(
            0,
            len(indices),
            batch_size
        ):

            batch_indices = indices[
                start:start + batch_size
            ]

            batch_x = []
            batch_y = []

            for index in batch_indices:

                frames = frames_extraction(
                    video_paths[index]
                )

                if frames is None:
                    continue

                batch_x.append(frames)

                batch_y.append(
                    labels[index]
                )

            if len(batch_x) > 0:

                yield (
                    np.array(batch_x),
                    np.array(batch_y)
                )

In [16]:
import os
import numpy as np

video_paths = []
video_labels = []

class_to_index = {
    class_name: index
    for index, class_name in enumerate(classes)
}

for class_name in classes:

    class_path = os.path.join(
        dataset_dir,
        class_name
    )

    for file_name in os.listdir(class_path):

        video_file_path = os.path.join(
            class_path,
            file_name
        )

        video_paths.append(video_file_path)

        video_labels.append(
            class_to_index[class_name]
        )

print("Total videos:", len(video_paths))

Total videos: 200


In [17]:
from sklearn.model_selection import train_test_split

train_paths, test_paths, train_labels, test_labels = train_test_split(
    video_paths,
    video_labels,
    test_size=0.2,
    random_state=42,
    stratify=video_labels
)

print("Training videos:", len(train_paths))
print("Testing videos:", len(test_paths))

Training videos: 160
Testing videos: 40


In [18]:
batch_size = 2

train_generator = data_generator(
    train_paths,
    train_labels,
    batch_size
)

test_generator = data_generator(
    test_paths,
    test_labels,
    batch_size
)

In [19]:
train_steps = int(
    np.ceil(
        len(train_paths) / batch_size
    )
)

test_steps = int(
    np.ceil(
        len(test_paths) / batch_size
    )
)

print("Train steps:", train_steps)
print("Test steps:", test_steps)

Train steps: 80
Test steps: 20


In [20]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

In [21]:
from tensorflow.keras.callbacks import ModelCheckpoint
checkpoint = ModelCheckpoint(
    filepath='best_crime_model.keras',  
    monitor='val_loss',  
    mode='min',  
    save_best_only=True, 
    verbose=1,
)

In [22]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_steps,
    validation_data=test_generator,
    validation_steps=test_steps,
    epochs=20,
    callbacks=[early_stopping,checkpoint]
)

C:\Users\PRAVALLIKA\AppData\Roaming\Python\Python313\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a generator. The generator is expected to yield already-shuffled data.
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3250 - loss: 1.3896  
Epoch 1: val_loss improved from None to 1.30378, saving model to best_crime_model.keras

Epoch 1: finished saving model to best_crime_model.keras
80/80 ━━━━━━━━━━━━━━━━━━━━ 582s 4s/step - accuracy: 0.3250 - loss: 1.3896 - val_accuracy: 0.4750 - val_loss: 1.3038
Epoch 2/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4563 - loss: 1.2283  
Epoch 2: val_loss improved from 1.30378 to 1.18069, saving model to best_crime_model.keras

Epoch 2: finished saving model to best_crime_model.keras
80/80 ━━━━━━━━━━━━━━━━━━━━ 265s 3s/step - accuracy: 0.4563 - loss: 1.2283 - val_accuracy: 0.5750 - val_loss: 1.1807
Epoch 3/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5188 - loss: 1.1121  
Epoch 3: val_loss improved from 1.18069 to 1.14039, saving model to best_crime_model.keras

Epoch 3: finished saving model to best_crime_model.keras
80/80 ━━━━━━━━━━━━━━━━━━━━ 270s 3s/step - accuracy: 0.5188 - loss: 1.

In [23]:
from tensorflow.keras.models import load_model

model = load_model("best_crime_model.keras")
print("Model loaded successfully!")
model.summary()

Model loaded successfully!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ time_distributed (TimeDistributed)   │ (None, 20, 1280)            │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 128)                 │         721,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │             260 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,239,345 (23.80 MB)

 Trainable params: 729,924 (2.78 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 1,459,850 (5.57 MB)

In [24]:
model.save(
    r"C:\Users\PRAVALLIKA\Downloads\crime2_efficientnet_lstm.keras"
)

In [25]:
test_video = r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos\Fighting\Fighting051_x264.mp4"

In [26]:
frames = frames_extraction(
    test_video
)

In [27]:
test_data = np.expand_dims(
    frames,
    axis=0
)

In [28]:
prediction = model.predict(
    test_data
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 73s 73s/step


In [29]:
prediction

array([[0.08134372, 0.78019035, 0.12521899, 0.01324693]], dtype=float32)

In [30]:
output=np.argmax(prediction)

In [31]:
classes[output]

'fighting'

In [78]:
test_video1=r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos\Explosion\Explosion048_x264.mp4"

In [91]:
test_video11=r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos\Explosion\Explosion042_x264.mp4"

In [93]:
frames = frames_extraction(
    test_video11
)

In [95]:
test_data1 = np.expand_dims(
    frames,
    axis=0
)
prediction1 = model.predict(
    test_data1
)
prediction1
output1=np.argmax(prediction1)
classes[output1]

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


'Explosion'

In [33]:
frames = frames_extraction(
    test_video1
)

In [34]:
test_data1 = np.expand_dims(
    frames,
    axis=0
)
prediction1 = model.predict(
    test_data1
)
prediction1
output1=np.argmax(prediction1)

In [35]:
prediction1 = model.predict(
    test_data1
)
output1=np.argmax(prediction1)
classes[output1]

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


array([[0.86385703, 0.10019158, 0.02259273, 0.01335869]], dtype=float32)

In [36]:
output1=np.argmax(prediction1)

In [37]:
classes[output1]

'Explosion'

In [79]:
test_video2=r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos\Fighting\Fighting044_x264.mp4"

In [100]:
test_video21=r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos\Fighting\Fighting007_x264.mp4"

In [102]:
test_video22=r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos\Fighting\Fighting004_x264.mp4"

In [39]:
frames = frames_extraction(test_video2)
test_data1 = np.expand_dims(frames,axis=0)
prediction1 = model.predict(test_data1)
output1=np.argmax(prediction1)
classes[output1]

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


'fighting'

In [80]:
test_video3=r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos\Shooting\Shooting054_x264.mp4"

In [85]:
test_video31=r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos\Shooting\Shooting018_x264.mp4"

In [89]:
test_video32=r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos\Shooting\Shooting007_x264.mp4"

In [97]:
test_video34=r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos\Shooting\Shooting004_x264.mp4"

In [98]:
frames = frames_extraction(test_video34)
test_data1 = np.expand_dims(frames,axis=0)
prediction1 = model.predict(test_data1)
output1=np.argmax(prediction1)
classes[output1]

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 984ms/step


'Shooting'

In [99]:
test_video4=r"C:\Users\PRAVALLIKA\Downloads\UCF_Crimes\UCF_Crimes\Videos\z_Normal_Videos_event\Normal_Videos_929_x264.mp4"

In [44]:
frames = frames_extraction(test_video4)
test_data1 = np.expand_dims(frames,axis=0)
prediction1 = model.predict(test_data1)
output1=np.argmax(prediction1)
classes[output1]

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


'z_Normal_Videos_event'

In [82]:
paths=[test_video1,test_video2,test_video3,test_video4]
for path in paths:
    load_video(path)

In [83]:
classes

['Explosion', 'Fighting', 'Shooting', 'Normal']

In [104]:
import cv2
import numpy as np


# FOUR TEST VIDEOS

video_paths = [
    test_video11,
    test_video22,
    test_video34,
    test_video4
]

classes = ['Explosion', 'fighting', 'Shooting', 'z_Normal_Videos_event']


# OPEN VIDEOS

caps = []

for path in video_paths:

    cap = cv2.VideoCapture(path)

    if not cap.isOpened():
        print("Video could not be opened:", path)

    caps.append(cap)

# 20-frame buffers
buffers = [[], [], [], []]


# PLAY 4 VIDEOS
while True:

    output = []
    for i in range(4):
        cap = caps[i]
        ret, frame = cap.read()
        
        if not ret or frame is None:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = cap.read()

        if not ret or frame is None:

            frame = np.zeros((300, 400, 3),dtype=np.uint8)

            cv2.putText(frame,"Video not available",(40, 150),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0, 0, 255),2)

            output.append(frame)

            continue

        # RESIZE
        frame = cv2.resize(
            frame,
            (400, 300)
        )

        
        # PREPROCESS FOR MODEL
        img = cv2.resize(
            frame,
            (224, 224)
        )

        img = img.astype(np.float32)

        buffers[i].append(img)

        if len(buffers[i]) > 20:
            buffers[i].pop(0)

    
        # PREDICTION

        label = "Analyzing"
        confidence = 0

        if len(buffers[i]) == 20:

            x = np.array(buffers[i])

            x = np.expand_dims(x,axis=0)

            pred = model.predict(x,verbose=0)[0]

            class_id = np.argmax(pred)

            label = classes[class_id]

            confidence = pred[class_id] * 100


        # DISPLAY PREDICTION
        cv2.rectangle(frame,(0, 0),(400, 45),(0, 0, 0),-1)

        cv2.putText(frame,f"{label} {confidence:.1f}%",(10, 30),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0, 255, 0),2)

        output.append(frame)

    # 2 x 2 GRID

    top = np.hstack((output[0], output[1]))

    bottom = np.hstack((output[2], output[3]))

    final = np.vstack((top, bottom))

    cv2.imshow("Crime Activity Detection",final)

    if cv2.waitKey(30) & 0xFF == ord("q"):
        break


# CLOSE

for cap in caps:
    cap.release()

cv2.destroyAllWindows()